<a href="https://colab.research.google.com/github/Nitesh-9009/TokensToTranslation/blob/main/secondWeek_SOC.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Week 2 - makemore (character level language model)

First week me micrograd banaya tha (apna chota autograd + MLP). Ab Karpathy ke **makemore** series ko follow kar raha hu.

Plan (part 1 se 5 tak):
1. bigram model (sirf counts se)
2. bigram ko neural net ki tarah (one-hot + ek linear layer)
3. MLP wala model (Bengio 2003 paper) -> character **embeddings**
4. **weight initialization** theek karna (starting loss kam karna)
5. **batch normalization**

dataset = 32k naam (names.txt).

In [ ]:
import urllib.request

# Karpathy ka names.txt seedha download kar lete hai
url = 'https://raw.githubusercontent.com/karpathy/makemore/master/names.txt'
urllib.request.urlretrieve(url, 'names.txt')

words = open('names.txt').read().splitlines()
print('total names:', len(words))
print('sample:', words[:8])
print('sabse chota / bada:', min(len(w) for w in words), max(len(w) for w in words))

## 1. Bigram model (counts wala)

Idea simple hai: har character ke baad kaunsa character aata hai uski ginti kar lo. `.` special token hai start/end ke liye.

In [ ]:
import torch

# vocab banao
chars = sorted(list(set(''.join(words))))
stoi = {s: i + 1 for i, s in enumerate(chars)}
stoi['.'] = 0                       # '.' ko index 0 diya
itos = {i: s for s, i in stoi.items()}
vocab_size = len(itos)
print('vocab size:', vocab_size)

# 27x27 count matrix
N = torch.zeros((27, 27), dtype=torch.int32)
for w in words:
    chs = ['.'] + list(w) + ['.']
    for ch1, ch2 in zip(chs, chs[1:]):
        N[stoi[ch1], stoi[ch2]] += 1

print('e ke baad kya aata hai (top counts):')
row = N[stoi['e']]
for ix in row.argsort(descending=True)[:5]:
    print(' ', itos[ix.item()], row[ix].item())

In [ ]:
# counts ko probability me convert karo. +1 -> model smoothing (kabhi 0 prob na aaye)
P = (N + 1).float()
P /= P.sum(1, keepdim=True)          # har row ka sum 1 ho jaye

# ab isse sample karke naam banate hai
g = torch.Generator().manual_seed(2147483647)
for _ in range(5):
    out = []
    ix = 0
    while True:
        p = P[ix]
        ix = torch.multinomial(p, num_samples=1, replacement=True, generator=g).item()
        if ix == 0:
            break
        out.append(itos[ix])
    print(''.join(out))

Naam thode ajeeb hai but bilkul random se better. Model kitna acha hai ye **negative log likelihood** se naapte hai (jitna kam utna acha).

In [ ]:
log_likelihood = 0.0
n = 0
for w in words:
    chs = ['.'] + list(w) + ['.']
    for ch1, ch2 in zip(chs, chs[1:]):
        prob = P[stoi[ch1], stoi[ch2]]
        log_likelihood += torch.log(prob)
        n += 1

nll = -log_likelihood / n
print('average negative log likelihood:', nll.item())

## 2. Same bigram, but neural net ki tarah

Ab counts matrix ki jagah ek `W` (27x27) weight seekhenge gradient descent se. Input character ko **one-hot** banate hai, `W` se multiply, phir softmax -> probabilities. Ye samajhne me time laga but end me loss almost bigram jitna hi aata hai (as expected).

In [ ]:
import torch.nn.functional as F

# training set of bigrams (x -> y)
xs, ys = [], []
for w in words:
    chs = ['.'] + list(w) + ['.']
    for ch1, ch2 in zip(chs, chs[1:]):
        xs.append(stoi[ch1])
        ys.append(stoi[ch2])
xs = torch.tensor(xs)
ys = torch.tensor(ys)
num = xs.nelement()
print('number of examples:', num)

g = torch.Generator().manual_seed(2147483647)
W = torch.randn((27, 27), generator=g, requires_grad=True)

In [ ]:
# gradient descent (poora dataset ek saath - simple hai)
for k in range(120):
    # forward
    xenc = F.one_hot(xs, num_classes=27).float()
    logits = xenc @ W                       # log-counts jaisa
    counts = logits.exp()
    probs = counts / counts.sum(1, keepdim=True)   # softmax
    loss = -probs[torch.arange(num), ys].log().mean() + 0.01 * (W ** 2).mean()

    # backward
    W.grad = None
    loss.backward()

    # update
    W.data += -50 * W.grad

print('final loss:', loss.item())
# note: ye ~2.45 aata hai, bigram counts wale ke bahut kareeb -> matlab net ne wahi seekh liya

## 3. MLP model (Bengio 2003) + character embeddings

Bigram me sirf 1 pichla character dekhte the. Ab context badhate hai (`block_size = 3` pichle characters) aur har character ko ek chhoti vector me embed karte hai. Yahi **character embedding** hai - lookup table `C`.

Dataset ko train/dev/test me todte hai (80/10/10).

In [ ]:
block_size = 3      # kitne pichle char dekhne hai


def build_dataset(words):
    X, Y = [], []
    for w in words:
        context = [0] * block_size          # shuru me sab '.'
        for ch in w + '.':
            ix = stoi[ch]
            X.append(context)
            Y.append(ix)
            context = context[1:] + [ix]     # sliding window
    return torch.tensor(X), torch.tensor(Y)


import random
random.seed(42)
random.shuffle(words)
n1 = int(0.8 * len(words))
n2 = int(0.9 * len(words))

Xtr, Ytr = build_dataset(words[:n1])
Xdev, Ydev = build_dataset(words[n1:n2])
Xte, Yte = build_dataset(words[n2:])
print('train:', Xtr.shape, '| dev:', Xdev.shape, '| test:', Xte.shape)

## 4 + 5. Weight init theek karna + Batch Normalization

Do bade fixes jo makemore part 3 me sikhaye:

- **weight init**: agar `W1`/`W2` bade random se init karo to shuruaat me loss bahut high (confidently galat), aur tanh saturate ho jata hai. Isliye `W1` ko kaiming init (`5/3 / sqrt(fan_in)`) aur `W2` ko chhota (`*0.01`) rakhte hai.
- **batchnorm**: hidden pre-activation ko har batch me normalize (mean 0, std 1) kar dete hai -> training stable ho jaati hai. Running mean/std bhi rakhte hai inference ke liye.

In [ ]:
n_embd = 10        # har char ka embedding size
n_hidden = 200     # hidden layer size
g = torch.Generator().manual_seed(2147483647)

C = torch.randn((vocab_size, n_embd), generator=g)
# kaiming-ish init -> tanh gain 5/3, fan_in = n_embd*block_size
W1 = torch.randn((n_embd * block_size, n_hidden), generator=g) * (5 / 3) / (n_embd * block_size) ** 0.5
b1 = torch.randn(n_hidden, generator=g) * 0.01
W2 = torch.randn((n_hidden, vocab_size), generator=g) * 0.01   # chhota -> starting loss kam
b2 = torch.randn(vocab_size, generator=g) * 0

# batchnorm ke parameters
bngain = torch.ones((1, n_hidden))
bnbias = torch.zeros((1, n_hidden))
bnmean_running = torch.zeros((1, n_hidden))
bnstd_running = torch.ones((1, n_hidden))

parameters = [C, W1, b1, W2, b2, bngain, bnbias]
print('total params:', sum(p.nelement() for p in parameters))
for p in parameters:
    p.requires_grad = True

In [ ]:
# training loop (minibatch SGD)
max_steps = 20000
batch_size = 32
lossi = []

for i in range(max_steps):
    # minibatch
    ix = torch.randint(0, Xtr.shape[0], (batch_size,), generator=g)
    Xb, Yb = Xtr[ix], Ytr[ix]

    # forward
    emb = C[Xb]                              # (batch, block_size, n_embd)
    embcat = emb.view(emb.shape[0], -1)      # flatten
    hpreact = embcat @ W1 + b1

    # ---- batchnorm ----
    bnmeani = hpreact.mean(0, keepdim=True)
    bnstdi = hpreact.std(0, keepdim=True)
    hpreact = bngain * (hpreact - bnmeani) / bnstdi + bnbias
    with torch.no_grad():
        bnmean_running = 0.999 * bnmean_running + 0.001 * bnmeani
        bnstd_running = 0.999 * bnstd_running + 0.001 * bnstdi
    # -------------------

    h = torch.tanh(hpreact)
    logits = h @ W2 + b2
    loss = F.cross_entropy(logits, Yb)

    # backward
    for p in parameters:
        p.grad = None
    loss.backward()

    # update (learning rate decay)
    lr = 0.1 if i < 10000 else 0.01
    for p in parameters:
        p.data += -lr * p.grad

    if i % 2000 == 0:
        print(f'{i:7d}/{max_steps:7d}: {loss.item():.4f}')
    lossi.append(loss.log10().item())

In [ ]:
# dev/train pe loss (batchnorm ke running stats use karke)
@torch.no_grad()
def split_loss(split):
    X, Y = {'train': (Xtr, Ytr), 'dev': (Xdev, Ydev), 'test': (Xte, Yte)}[split]
    emb = C[X]
    embcat = emb.view(emb.shape[0], -1)
    hpreact = embcat @ W1 + b1
    hpreact = bngain * (hpreact - bnmean_running) / bnstd_running + bnbias
    h = torch.tanh(hpreact)
    logits = h @ W2 + b2
    loss = F.cross_entropy(logits, Y)
    print(split, loss.item())


split_loss('train')
split_loss('dev')

## Naam generate karo (MLP model se)

Ab dekho bigram wale se kaafi behtar naam aate hai.

In [ ]:
g = torch.Generator().manual_seed(2147483647 + 10)

for _ in range(15):
    out = []
    context = [0] * block_size
    while True:
        emb = C[torch.tensor([context])]
        embcat = emb.view(1, -1)
        hpreact = embcat @ W1 + b1
        hpreact = bngain * (hpreact - bnmean_running) / bnstd_running + bnbias
        h = torch.tanh(hpreact)
        logits = h @ W2 + b2
        probs = F.softmax(logits, dim=1)
        ix = torch.multinomial(probs, num_samples=1, generator=g).item()
        context = context[1:] + [ix]
        if ix == 0:
            break
        out.append(itos[ix])
    print(''.join(out))

## Week 2 summary (mera notes)

- bigram counts -> simplest LM, nll ~2.45
- wahi bigram neural net se bhi ban gaya (same loss) -> gradient descent kaam karta hai
- MLP + **character embeddings** se context use kar paye, loss ~2.1 ke aas paas
- **weight init** galat ho to shuruaati loss high aur tanh dead ho jata hai -> kaiming init + chhota last layer
- **batchnorm** ne training smooth ki

Next week: RNN, LSTM, attention, transformers.